### Task 1: classify Parkinson's vs control

In [ ]:
# Import libraries for data handling, model training, preprocessing, and evaluation

import json
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

#### 2. Define File Paths and Output Directories


In [ ]:
# define paths to the shared feature dataset and saved data splits.

ROOT = Path.cwd().resolve()  # get the current directory as an absolute path
if not (ROOT / "src").exists():
    ROOT = ROOT.parent  # move up one directory if src/ is not found

assert (ROOT / "src").exists(), f"Cannot locate project root from {Path.cwd()}"

# define paths to the input data files
FEATURES_PATH = ROOT / "data" / "features.csv"
SPLITS_PATH = ROOT / "data" / "splits.json"

# define the output directory for Task 1 results
OUT = ROOT / "results" / "task1"

# create folders for predictions, trained model weights, and figures
(OUT / "predictions").mkdir(parents=True, exist_ok=True)
(OUT / "weights").mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(parents=True, exist_ok=True)

# verify the project root and check whether the input files exist
print("Project root:", ROOT)
print("Features exist:", FEATURES_PATH.exists())
print("Splits exist:", SPLITS_PATH.exists())

#### 3. Load the Predefined Data Splits
load the saved subject-level data splits from splits.json

In [ ]:
# load the predefined Task 1 training, validation, and test subject IDs.

with open(SPLITS_PATH, "r") as f:
    assignment = json.load(f)["task1"]

# store the held-out test subjects and five cross-validation folds
test_subjects = set(assignment["test"])
folds = assignment["folds"]

# check the number and structure of the saved splits
print("Number of folds:", len(folds))
print("Number of held-out test subjects:", len(test_subjects))
print("First fold keys:", folds[0].keys())

# verify that the expected five-fold setup is present
assert len(folds) == 5, "Expected five cross-validation folds"
assert len(test_subjects) > 0, "The test set is empty"
assert all("train" in fold and "val" in fold for fold in folds), \
    "Each fold must contain training and validation subject IDs"

#### 4. Load and Inspect the Feature Dataset

In [ ]:
# inspect feature table

features = pd.read_csv(FEATURES_PATH)

print("Feature table dimensions:", features.shape)
print("Feature columns:", features.columns.tolist())
features.head()
# features.dtypes

#### 5. Function to split data

In [ ]:
# select training and validation rows based on the predefined subject IDs.
# confirm the subject column name once features.csv is available.
def get_fold_data(features, train_ids, val_ids):
    train_mask = features["subject"].isin(train_ids)
    val_mask = features["subject"].isin(val_ids)

    train_df = features.loc[train_mask]
    val_df = features.loc[val_mask]

    return train_df, val_df



#### 6. Select Predictors and Diagnosis Target

In [ ]:
# select numerical predictors and the diagnosis target.

target_column = "label"  # column containing the Parkinson's disease diagnosis
subject_column = "subject"  # column identifying each participant

# exclude identifiers, the target, and other non-predictive or potentially leaking columns
excluded_columns = {
    subject_column, target_column, "file", "filename", "walk_file",
    "walk", "study", "group", "UPDRSM"
}

# select numerical columns that are not in the exclusion list
feature_columns = [
    col for col in features.columns
    if col not in excluded_columns
    and pd.api.types.is_numeric_dtype(features[col])
]

# separate the predictor variables (X) from the target labels (y)
X = features[feature_columns]
y = features[target_column]

# inspect the selected predictors and diagnosis labels
print("Predictor matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Diagnosis labels:", y.unique())
print("Selected features:", feature_columns)

#### 7. Model Evaluation Function

In [ ]:
# calculate classification metrics for one set of predictions.

def evaluate_model(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, pos_label="Pt"),
        "roc_auc": roc_auc_score(
            (pd.Series(y_true) == "Pt").astype(int),
            y_prob
        )
    }

In [24]:
# initialise the classification models to compare.

models = {
    "Majority Baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=1)
    ),
    "Random Forest": RandomForestClassifier(random_state=1)
}